# Libraries

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from functions import *
import joblib
import numpy as np
import pandas as pd
from UQpy.distributions import Normal, JointIndependent

# Run PCE

### Realizations of $R$ and $S$

In [8]:
n_samples = 1000
n_latent_samples = 5000
r   = Normal(loc = 5., scale=0.8)
s   = Normal(loc = 2., scale=0.6)
joint = JointIndependent(marginals=[r, s])
x = joint.rvs(n_samples)
x

array([[4.36264005, 1.35547433],
       [4.66324255, 2.04237452],
       [4.82314207, 1.71449781],
       ...,
       [4.80302793, 1.14660807],
       [4.57200426, 2.44776976],
       [5.16099522, 1.03011245]])

In [9]:
# Load the PCE metamodels
times = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
pces = []
for i in times:
    pce_i = joblib.load(f'pce_metamodel_{int(i)}.pkl')
    pces.append(pce_i)

In [10]:
lambdas = []
for i in range(len(times)):
    lambdas.append(pces[i].predict(x))

In [11]:
dados_completos = []
for i in range(len(times)):
    t_atual = times[i]  # Pega o inteiro do tempo (ex: 0, 5, 10...)
    
    # 1. Descobre quantas amostras (linhas) tem no x
    n_linhas = x.shape[0] 
    
    # 2. Cria uma coluna vertical repetindo esse tempo
    coluna_tempo = np.full((n_linhas, 1), t_atual)
    
    # 3. Concatena: [ X | Lambda_i | Tempo ]
    bloco = np.concatenate((x, lambdas[i], coluna_tempo), axis=1)
    dados_completos.append(bloco)
matriz_final = np.vstack(dados_completos)
nomes_colunas = ['R', 'S', 'lambda1', 'lambda2', 'lambda3', 'lambda4', 'Time']
df = pd.DataFrame(matriz_final, columns=nomes_colunas)
df

,R,S,lambda1,lambda2,lambda3,lambda4,Time
0,4.362640,1.355474,3.013870,8.248070,0.114479,0.146328,0.0
1,4.663243,2.042375,2.631957,6.287037,0.101774,0.158632,0.0
2,4.823142,1.714498,3.117534,6.940378,0.109569,0.150951,0.0
3,5.124100,1.677622,3.454972,6.841155,0.112623,0.147992,0.0
4,5.942307,2.337918,3.616274,5.302841,0.106950,0.155045,0.0
...,...,...,...,...,...,...,...
10995,5.967390,2.112948,-0.310058,7.072210,0.081844,0.182620,100.0
10996,5.245746,2.093981,-0.507806,7.074219,0.081326,0.185012,100.0
10997,4.803028,1.146608,0.301254,12.961591,0.088217,0.176332,100.0
10998,4.572004,2.447770,-1.061881,6.152034,0.080308,0.186426,100.0


In [ ]:
df.to_excel('dataset_lambda_from_pce.xlsx', index=False)